# Encapsulation and Shared References

CSC-239 · Module 5 · Lesson 2 of 3

You can define objects with fields, constructors and instance methods. Now you will give those objects rules for accepting changes and examine what happens when references share an object.

Select the **Java** kernel in your Workspace. Restart the kernel and run the ordinary cells in order. Supporting examples use distinct class names so their class-wide counts stay separate. Before replaying a complete counted fixture or testing a hidden answer, restart the kernel: repeating constructor calls can increase its existing static count.


## Learning Goals

- Protect a state rule with private fields and guarded public operations.
- Distinguish reference reassignment, shared-object mutation and class-wide state.


## Why This Matters

A supply bin should not give out more items than it contains. If two parts of a program refer to that bin, both should observe the same accepted change.


## Check Your Starting Point

Explain the difference between two new expressions and two variable names. Recall why changing a numeric parameter does not replace its caller’s variable, and how a method can return false for an input that does not meet a rule.

**My explanation:**


## Concept

### Keep changes behind operations

**Encapsulation** means keeping an object’s state behind operations that control how it changes. The previous pass model let lesson code read its fields directly. A supply bin needs a stronger boundary: callers may ask for items, but they should not assign a negative stock count.

**Access control** uses modifiers such as private and public to restrict or expose members. For the top-level classes in this lesson, a private field can be used by code inside its class, while other lesson code uses the class’s public operations. A public method gives callers an intended entry point; it does not make the private field public.

A **state invariant** is a rule that stays true after each accepted operation. Our bin’s rule is stock >= 0. The constructor accepts an initially nonnegative count. That is a **precondition**: a rule callers must satisfy before the operation begins. The constructor does not check that rule for you. Later exception lessons will show how a constructor can reject an invalid input.

Here is a complete guarded object and caller:

```java
class GuardedBin {
    private int stock;
    public GuardedBin(int stock) {
        this.stock = stock;
    }
    public boolean take(int amount) {
        if (amount <= 0 || amount > stock) {
            return false;
        }
        stock = stock - amount;
        return true;
    }
    public int getStock() {
        return stock;
    }
}
GuardedBin bin = new GuardedBin(4);
System.out.println(bin.take(2));
System.out.println(bin.take(3));
System.out.println(bin.getStock());
```

The output is true, false, and 2 on separate lines. The first request subtracts two items. The second request is too large, so the early return happens before any subtraction. getStock returns the remaining count without changing it.

An exact request for all remaining stock is valid. A zero or negative request is invalid. The rejection result and unchanged stock are both useful evidence: checking only the returned boolean could miss an accidental state change.

The following is an intentionally invalid access example, shown for explanation only. Do not add it to a runnable cell:

```text
bin.stock = -1;  // Rejected: stock is private to GuardedBin.
```

Use bin.take(...) and inspect its returned result. Making a field private works with the class’s operation rules; private alone does not guarantee that every method preserves a sensible state.

### Distinguish a second object from a second reference

**Aliasing** means two reference variables refer to the same object. Assigning one reference variable to another copies the reference value. It does not create a new object or copy the object’s fields.

```java
class SharedCounter {
    private int count;
    public SharedCounter(int count) {
        this.count = count;
    }
    public void addOne() {
        count = count + 1;
    }
    public int getCount() {
        return count;
    }
}
SharedCounter original = new SharedCounter(2);
SharedCounter alias = original;
alias.addOne();
System.out.println(original.getCount());
System.out.println(alias.getCount());
```

Both lines print 3. There is one object and two ways to reach it. Calling addOne through either reference changes that object. This differs from the previous lesson’s two separate new expressions, which created two independent objects.

### A copied reference can still reach a mutable object

**Mutation** means changing an existing object’s state. Java still passes arguments by value when the value is a reference. A parameter receives a copy of that reference and can reach the same object as the caller.

Reassigning the parameter changes only the parameter. Calling a mutating method through the parameter changes the shared object. This complete example separates those two actions:

```java
class SharedCounter {
    private int count;
    public SharedCounter(int count) {
        this.count = count;
    }
    public void addOne() {
        count = count + 1;
    }
    public int getCount() {
        return count;
    }
}
class CounterTools {
    static void replaceLocal(SharedCounter counter) {
        counter = new SharedCounter(9);
    }
    static void addThroughReference(SharedCounter counter) {
        counter.addOne();
    }
}
SharedCounter original = new SharedCounter(2);
CounterTools.replaceLocal(original);
System.out.println(original.getCount());
CounterTools.addThroughReference(original);
System.out.println(original.getCount());
```

This prints 2 and then 3. replaceLocal creates another object and assigns its reference to the local parameter; original still refers to the first object. addThroughReference changes the object both caller and parameter can reach. Do not describe this as Java passing objects by reference: the argument value is copied in both calls.

The field update is a side effect. You already practiced the distinction between a pure calculation and an action that changes state outside a method’s local variables.

### Separate class state from instance state

A **static field** belongs to the class and is shared across its instances. An instance field belongs separately to each object. A bin’s stock is instance state; a count of how many bins have been constructed can be class state.

```java
class CountedBin {
    private int stock;
    private static int created = 0;
    public CountedBin(int stock) {
        this.stock = stock;
        created = created + 1;
    }
    public int getStock() {
        return stock;
    }
    public static int getCreated() {
        return created;
    }
}
CountedBin first = new CountedBin(6);
CountedBin second = new CountedBin(1);
System.out.println(first.getStock());
System.out.println(second.getStock());
System.out.println(CountedBin.getCreated());
```

The output is 6, 1, and 2. Each constructor sets one object’s stock and increments the same created field. Call the static reader through CountedBin because its result describes the class’s shared count. That method has no current instance, so it cannot directly choose one bin’s stock.

The counter records constructions, not the number of bins still in use. A new kernel begins a new lesson session. Repeating an identical class definition in IJava does not reliably reset its static fields. If you rerun its object-creation code in the same kernel, the count can increase again.

The ordinary runnable examples use distinct class names to keep their class-wide counts separate when you run the notebook in order. A different class name means a different class and a different static field. Before replaying a complete counted example or running one complete hidden answer, restart the kernel and use that example's full definition and caller. Do not claim that retyping a class is a reset operation.


## Video Demonstration

Watch a valid request change one bin and an excessive request leave it unchanged. Distinguish the stock in each object from the single count stored by the class.

<video controls preload="metadata" width="960">
  <source src="media/02_encapsulation_and_shared_references/demo.mp4" type="video/mp4">
  <track kind="captions" src="media/02_encapsulation_and_shared_references/captions.vtt" srclang="en" label="English">
  Your browser does not support embedded video.
</video>

[Read the encapsulation and shared references demonstration transcript](media/02_encapsulation_and_shared_references/transcript.md).


## Worked Example

**Subgoal 1: protect the stock rule.** Use a private field and return false before any invalid subtraction.

**Subgoal 2: track two kinds of state.** Keep each bin’s stock separate and increment the shared construction count.

**Subgoal 3: inspect accepted and rejected requests.** Use the boolean result and the remaining stock together.


In [ ]:
class SupplyBin {
    private int stock;
    private static int binsCreated = 0;
    public SupplyBin(int stock) {
        this.stock = stock;
        binsCreated = binsCreated + 1;
    }
    public boolean take(int amount) {
        if (amount <= 0 || amount > stock) {
            return false;
        }
        stock = stock - amount;
        return true;
    }
    public int getStock() {
        return stock;
    }
    public static int getBinsCreated() {
        return binsCreated;
    }
}
SupplyBin pens = new SupplyBin(5);
SupplyBin maps = new SupplyBin(2);
System.out.println("Take 3: " + pens.take(3));
System.out.println("Take 4: " + pens.take(4));
System.out.println("Pens: " + pens.getStock());
System.out.println("Maps: " + maps.getStock());
System.out.println("Bins: " + SupplyBin.getBinsCreated());


Expected output:

```text
Take 3: true
Take 4: false
Pens: 2
Maps: 2
Bins: 2
```

Taking three from five succeeds and leaves two pens. Taking four then fails without changing the stock. The maps bin still has two items. Both constructor calls increment the class-wide count, so two bins have been created.


## Predict, Run, Trace, and Explain

### Predict guarded requests and two kinds of state

Read the complete program without running it. Predict every output line. For each request, decide whether the rejection guard returns before the subtraction and what stock remains afterward. Explain what belongs to each object and what the single binsCreated field counts. Record your prediction before running the next cell or opening the answer.

My predicted complete output:

The stock before and after each request:

Which request is accepted and which is rejected:

What each object stores separately:

What binsCreated measures:


In [ ]:
class PredictionBin {
    private int stock;
    private static int binsCreated = 0;
    public PredictionBin(int stock) {
        this.stock = stock;
        binsCreated = binsCreated + 1;
    }
    public boolean take(int amount) {
        if (amount <= 0 || amount > stock) {
            return false;
        }
        stock = stock - amount;
        return true;
    }
    public int getStock() {
        return stock;
    }
    public static int getBinsCreated() {
        return binsCreated;
    }
}
PredictionBin badges = new PredictionBin(6);
PredictionBin cards = new PredictionBin(3);
System.out.println("Take 2: " + badges.take(2));
System.out.println("Take 5: " + badges.take(5));
System.out.println("Badges: " + badges.getStock());
System.out.println("Cards: " + cards.getStock());
System.out.println("Bins: " + PredictionBin.getBinsCreated());


If this fixture has already run, restart the Java kernel first. Then run the complete cell once. Keep your original prediction and record the actual output beside it. Find the first matching or differing part of your trace. Explain why the returned boolean and the remaining stock are both necessary checks. Explain why the rejected request leaves the badge stock unchanged and why the cards object and construction counter have their reported values.

My original prediction:

My actual complete output:

The first matching or differing point:

Why the rejected request makes no state change:

Why both the boolean and the stock matter:

Why the other object stays separate:

Why the construction count has its value:

### Trace access, invariant checks and shared references

Trace the two requests in the prediction program. Record stock before the call, whether the rejection condition is true, the returned boolean, and stock afterward. State the state invariant and the constructor precondition. Identify the private instance field, the private static field, and the public operations used by the caller. Explain why getStock can return a value without making its field public. Then complete the three separate checks below to distinguish aliasing, mutation through a parameter, and reassignment of a parameter. Each runnable check uses a distinct class name for its first run. Before replaying any counted check, restart the Java kernel and run that complete cell once; repeating an identical class definition is not a reliable counter reset.

| Request | Badge stock before | Rejection condition true? | Returned boolean | Badge stock after | Cards stock after | Construction count |
|---|---|---|---|---|---|---|
| badges.take(2) | | | | | | |
| badges.take(5) | | | | | | |


The nonnegative-stock state invariant:

The constructor input precondition:

Why the constructor does not promise to reject a negative starting count:

The instance field and the static field:

The caller’s public operations:

Why an outside assignment to the private stock field is not permitted:

Why the guard must run before subtraction:

My observed output agrees with the trace because:

<details>
<summary>Show answer</summary>

The two new expressions create separate bins with stock 6 and 3. Each constructor also increments the same binsCreated field, giving a construction count of 2. Taking 2 badges is allowed, so stock falls from 6 to 4 and take returns true. Taking 5 then exceeds the remaining 4, so the early return produces false before any subtraction. The badge stock remains 4; the cards object remains at 3. The public readers expose observations without allowing callers to assign the private fields. Starting from a nonnegative count and accepting only positive amounts no larger than stock preserves the rule stock >= 0. At the first request, amount is 2, stock before the call is 6, and the rejection condition is false. At the second request, amount is 5 and stock before the call is 4, so the rejection condition is true. In both cases binsCreated remains 2 because take does not run a constructor. Each stock field belongs to one instance; binsCreated belongs to the class. Private access supports encapsulation, but the guard and the order of statements are what preserve the state rule. The constructor assumes its supplied starting count is nonnegative; it does not validate that assumption.

```java
class PredictionBin {
    private int stock;
    private static int binsCreated = 0;
    public PredictionBin(int stock) {
        this.stock = stock;
        binsCreated = binsCreated + 1;
    }
    public boolean take(int amount) {
        if (amount <= 0 || amount > stock) {
            return false;
        }
        stock = stock - amount;
        return true;
    }
    public int getStock() {
        return stock;
    }
    public static int getBinsCreated() {
        return binsCreated;
    }
}
PredictionBin badges = new PredictionBin(6);
PredictionBin cards = new PredictionBin(3);
System.out.println("Take 2: " + badges.take(2));
System.out.println("Take 5: " + badges.take(5));
System.out.println("Badges: " + badges.getStock());
System.out.println("Cards: " + cards.getStock());
System.out.println("Bins: " + PredictionBin.getBinsCreated());
```

Expected output:

```text
Take 2: true
Take 5: false
Badges: 4
Cards: 3
Bins: 2
```

Common error: Subtracting an excessive request even when the method returns false. Combining the stock of separate bins. Counting take calls as constructions. Assuming private alone proves that every method preserves a valid stock count.

</details>


### Trace two references to one protected object

Predict all four output lines before running. Count the new expressions, identify which object original and alias can reach, and decide whether the assignment creates another bin. Run the complete cell and compare the two reader results. Explain why an accepted request through alias is visible through original and why private still prevents direct outside assignment to stock. This cell includes all needed definitions. Restart the Java kernel before replaying it, including after an earlier Run All; then run this complete cell once.

My predicted complete output:

The number of new expressions and constructed objects:

What value is copied into alias:

The object changed by alias.take(2):

My actual complete output:

Why both reader values agree:

Why the construction count differs from the number of reference variables:

I restarted before replaying this counted fixture:


In [ ]:
class AliasBin {
    private int stock;
    private static int binsCreated = 0;
    public AliasBin(int stock) {
        this.stock = stock;
        binsCreated = binsCreated + 1;
    }
    public boolean take(int amount) {
        if (amount <= 0 || amount > stock) {
            return false;
        }
        stock = stock - amount;
        return true;
    }
    public int getStock() {
        return stock;
    }
    public static int getBinsCreated() {
        return binsCreated;
    }
}
AliasBin original = new AliasBin(7);
AliasBin alias = original;
System.out.println("Take 2: " + alias.take(2));
System.out.println("Original: " + original.getStock());
System.out.println("Alias: " + alias.getStock());
System.out.println("Bins: " + AliasBin.getBinsCreated());


Record your own post-run explanation before opening the answer.

<details>
<summary>Show answer</summary>

There is only one new AliasBin expression. Assigning original to alias copies the reference value, so both variables reach the same object; it does not copy its fields or call a constructor. The take call through alias changes that object from 7 to 5. Both readers therefore report 5. The class counter remains 1 because creating another reference did not create another bin. The private field still changes only through the public take operation, which enforces the input rule.

```java
class AliasBin {
    private int stock;
    private static int binsCreated = 0;
    public AliasBin(int stock) {
        this.stock = stock;
        binsCreated = binsCreated + 1;
    }
    public boolean take(int amount) {
        if (amount <= 0 || amount > stock) {
            return false;
        }
        stock = stock - amount;
        return true;
    }
    public int getStock() {
        return stock;
    }
    public static int getBinsCreated() {
        return binsCreated;
    }
}
AliasBin original = new AliasBin(7);
AliasBin alias = original;
System.out.println("Take 2: " + alias.take(2));
System.out.println("Original: " + original.getStock());
System.out.println("Alias: " + alias.getStock());
System.out.println("Bins: " + AliasBin.getBinsCreated());
```

Expected output:

```text
Take 2: true
Original: 5
Alias: 5
Bins: 1
```

Common error: Treating alias = original as a new object construction. Copying the current stock value instead of tracing the shared object. Counting reference variables in the static construction counter.

</details>


### Trace mutation through a copied reference parameter

Predict every output line before running. At the helper call, identify the value copied into bin and the object reached by both the parameter and original. Trace the public take operation, then run the complete cell. Explain why changing the object remains visible after the helper returns even though Java copied the argument value. Compare this with changing only an int parameter in the previous lesson. This cell includes all needed definitions. Restart the Java kernel before replaying it, including after an earlier Run All; then run this complete cell once.

My predicted complete output:

The value copied into the bin parameter:

The object reached by caller and parameter:

The state changed inside the helper:

My actual complete output:

Why the caller can observe this mutation:

Why Java still passes the argument by value:

How this differs from changing a numeric parameter alone:

I restarted before replaying this counted fixture:


In [ ]:
class MutationBin {
    private int stock;
    private static int binsCreated = 0;
    public MutationBin(int stock) {
        this.stock = stock;
        binsCreated = binsCreated + 1;
    }
    public boolean take(int amount) {
        if (amount <= 0 || amount > stock) {
            return false;
        }
        stock = stock - amount;
        return true;
    }
    public int getStock() {
        return stock;
    }
    public static int getBinsCreated() {
        return binsCreated;
    }
}
class MutationTools {
    static void takeThroughParameter(MutationBin bin) {
        System.out.println("Take 2: " + bin.take(2));
    }
}
MutationBin original = new MutationBin(7);
MutationTools.takeThroughParameter(original);
System.out.println("Original: " + original.getStock());
System.out.println("Bins: " + MutationBin.getBinsCreated());


Record your own post-run explanation before opening the answer.

<details>
<summary>Show answer</summary>

When takeThroughParameter is called, Java copies the reference value held by original into the parameter bin. Both references reach the same existing MutationBin. Calling bin.take(2) changes that shared object from 7 to 5 and prints the true result. After the helper returns, original still reaches that changed object. No new expression occurs inside the helper, so binsCreated remains 1. Java passes the reference value by value; the object itself is not copied, and assigning the parameter would not assign the caller’s original variable.

```java
class MutationBin {
    private int stock;
    private static int binsCreated = 0;
    public MutationBin(int stock) {
        this.stock = stock;
        binsCreated = binsCreated + 1;
    }
    public boolean take(int amount) {
        if (amount <= 0 || amount > stock) {
            return false;
        }
        stock = stock - amount;
        return true;
    }
    public int getStock() {
        return stock;
    }
    public static int getBinsCreated() {
        return binsCreated;
    }
}
class MutationTools {
    static void takeThroughParameter(MutationBin bin) {
        System.out.println("Take 2: " + bin.take(2));
    }
}
MutationBin original = new MutationBin(7);
MutationTools.takeThroughParameter(original);
System.out.println("Original: " + original.getStock());
System.out.println("Bins: " + MutationBin.getBinsCreated());
```

Expected output:

```text
Take 2: true
Original: 5
Bins: 1
```

Common error: Claiming Java switches to pass by reference for objects. Assuming copying a reference duplicates the whole object. Expecting a field mutation to disappear when the parameter goes out of use.

</details>


### Trace local reassignment and a shared construction count

Predict every output line before running. Trace the copied reference before and after bin = new ReplacementBin(9). Decide whether original changes and whether another constructor runs. Run the complete cell. Explain why the original stock can stay the same while the static construction count changes. Compare the reassignment with the preceding mutation example; each cell starts with its own full class definition and fresh setup. This cell includes all needed definitions. Restart the Java kernel before replaying it, including after an earlier Run All; then run this complete cell once.

My predicted complete output:

What bin refers to before reassignment:

What bin refers to afterward:

What original still refers to:

The number of constructor calls:

My actual complete output:

Why the original stock is unchanged:

Why the class counter changes:

The difference between changing an object and reassigning a local parameter:

I restarted before replaying this counted fixture:


In [ ]:
class ReplacementBin {
    private int stock;
    private static int binsCreated = 0;
    public ReplacementBin(int stock) {
        this.stock = stock;
        binsCreated = binsCreated + 1;
    }
    public boolean take(int amount) {
        if (amount <= 0 || amount > stock) {
            return false;
        }
        stock = stock - amount;
        return true;
    }
    public int getStock() {
        return stock;
    }
    public static int getBinsCreated() {
        return binsCreated;
    }
}
class ReplacementTools {
    static void replaceLocal(ReplacementBin bin) {
        bin = new ReplacementBin(9);
    }
}
ReplacementBin original = new ReplacementBin(7);
ReplacementTools.replaceLocal(original);
System.out.println("Original: " + original.getStock());
System.out.println("Bins: " + ReplacementBin.getBinsCreated());


Record your own post-run explanation before opening the answer.

<details>
<summary>Show answer</summary>

The parameter bin initially receives a copy of original’s reference. The assignment inside replaceLocal creates a second ReplacementBin with stock 9 and makes only the local parameter refer to it. It does not assign original or mutate the first object. The caller therefore still reads stock 7. The new constructor did run, so the shared construction counter becomes 2. That count records constructions, not the number of variables or objects still reachable after the helper returns. Java copies the argument value in both helper examples: mutation reaches the shared object, while reassignment replaces only the local variable’s value.

```java
class ReplacementBin {
    private int stock;
    private static int binsCreated = 0;
    public ReplacementBin(int stock) {
        this.stock = stock;
        binsCreated = binsCreated + 1;
    }
    public boolean take(int amount) {
        if (amount <= 0 || amount > stock) {
            return false;
        }
        stock = stock - amount;
        return true;
    }
    public int getStock() {
        return stock;
    }
    public static int getBinsCreated() {
        return binsCreated;
    }
}
class ReplacementTools {
    static void replaceLocal(ReplacementBin bin) {
        bin = new ReplacementBin(9);
    }
}
ReplacementBin original = new ReplacementBin(7);
ReplacementTools.replaceLocal(original);
System.out.println("Original: " + original.getStock());
System.out.println("Bins: " + ReplacementBin.getBinsCreated());
```

Expected output:

```text
Original: 7
Bins: 2
```

Common error: Assuming assignment to bin also assigns the caller’s original variable. Ignoring the constructor because its reference stays local to the helper. Treating the counter as a count of variables or currently reachable objects.

</details>


## Guided Practice

Complete these tasks in order. The intentionally empty code cells are safe to run, but remain unfinished until you write and check your code.


### Complete a protected ticket operation

The displayed draft is incomplete and for reading only. Copy it into the empty work cell and replace all four placeholders. Choose FIELD_ACCESS from private or public so outside callers cannot assign the field. Choose REJECT_TEST from `amount <= 0 || amount > remaining` or `amount <= 0 && amount > remaining`. Choose UPDATED_REMAINING from `remaining - amount` or `remaining + amount`. Choose SUCCESS_RESULT from true or false. Keep the rest unchanged. Predict the exact request for all three tickets and the next request from the empty box. Run the completed program. Explain how the private field, public operations and guard work together, and why zero remaining is valid.

This sample is for repair:

```java
class TicketBox {
    FIELD_ACCESS int remaining;
    public TicketBox(int remaining) {
        this.remaining = remaining;
    }
    public boolean issue(int amount) {
        if (REJECT_TEST) {
            return false;
        }
        remaining = UPDATED_REMAINING;
        return SUCCESS_RESULT;
    }
    public int getRemaining() {
        return remaining;
    }
}
TicketBox tickets = new TicketBox(3);
System.out.println("Issue 3: " + tickets.issue(3));
System.out.println("Issue 1: " + tickets.issue(1));
System.out.println("Remaining: " + tickets.getRemaining());
```


My four completed choices:

My predicted complete output:

My actual complete output:

Why either invalid condition must reject:

Why the guard runs before the subtraction:

Why the exact request can leave zero:

Why the public reader does not expose a public field:

<details>
<summary>Show answer</summary>

FIELD_ACCESS is private, so outside lesson code uses the public operations instead of assigning remaining. REJECT_TEST is amount <= 0 || amount > remaining: either a nonpositive request or an excessive one must return false. UPDATED_REMAINING is remaining - amount, and SUCCESS_RESULT is true. The guard runs before subtraction. With 3 tickets, requesting 3 is valid and leaves zero. Requesting 1 then returns false without changing zero. getRemaining is a public reader; it returns the private field’s value without changing it or making the field public. The constructor receives the allowed nonnegative starting count 3.

```java
class TicketBox {
    private int remaining;
    public TicketBox(int remaining) {
        this.remaining = remaining;
    }
    public boolean issue(int amount) {
        if (amount <= 0 || amount > remaining) {
            return false;
        }
        remaining = remaining - amount;
        return true;
    }
    public int getRemaining() {
        return remaining;
    }
}
TicketBox tickets = new TicketBox(3);
System.out.println("Issue 3: " + tickets.issue(3));
System.out.println("Issue 1: " + tickets.issue(1));
System.out.println("Remaining: " + tickets.getRemaining());
```

Expected output:

```text
Issue 3: true
Issue 1: false
Remaining: 0
```

Common error: Using public for the field when direct outside assignment must be restricted. Combining rejection conditions with && and allowing one invalid condition through. Adding the issued amount instead of subtracting it. Rejecting an exact request because it leaves zero.

</details>


### Replace an alias with a separate object

This complete ModificationBin starter has the same behavior as the earlier alias check and uses a separate class name. For every run in this task, restart the Java kernel first, then run only the complete work cell once. Predict and run the starter. Change only `ModificationBin alias = original;` to `ModificationBin alias = new ModificationBin(7);`. Keep the class, other statements and variable names unchanged. Predict the result, restart the kernel and run the complete modified cell. Explain which outputs differ and why the variable name alias does not decide whether two references share an object. Then insert `System.out.println("Take 1 from original: " + original.take(1));` immediately after the Take 2 print statement. Predict the result, restart the kernel and run the whole cell again. Explain which object changes. Restore the original reference-copy assignment and remove the extra call. Restart once more before checking the restored starter.


In [ ]:
class ModificationBin {
    private int stock;
    private static int binsCreated = 0;
    public ModificationBin(int stock) {
        this.stock = stock;
        binsCreated = binsCreated + 1;
    }
    public boolean take(int amount) {
        if (amount <= 0 || amount > stock) {
            return false;
        }
        stock = stock - amount;
        return true;
    }
    public int getStock() {
        return stock;
    }
    public static int getBinsCreated() {
        return binsCreated;
    }
}
ModificationBin original = new ModificationBin(7);
ModificationBin alias = original;
System.out.println("Take 2: " + alias.take(2));
System.out.println("Original: " + original.getStock());
System.out.println("Alias: " + alias.getStock());
System.out.println("Bins: " + ModificationBin.getBinsCreated());


My predicted and actual starter output:

My predicted and actual output after adding the second new expression:

Which object each variable now reaches:

Why the construction count changes:

My predicted and actual output after the extra call on original:

Why the second object keeps its own stock:

My output after restoring the original alias starter:

How I ensured a fresh static count before each variant:

<details>
<summary>Show answer</summary>

Replacing the reference-copy assignment with a second new ModificationBin(7) expression creates another object and invokes another constructor. The variable named alias now reaches a separate bin; its name does not force it to be an alias. Taking 2 through that reference changes only the second object to 5. original still reaches the first object with stock 7, and binsCreated is 2. The extra original.take(1) call changes only the first object to 6; the second stays at 5. Both use the same protected operations, while their instance fields stay separate. The static counter is shared and records the two constructions.

```java
class ModificationBin {
    private int stock;
    private static int binsCreated = 0;
    public ModificationBin(int stock) {
        this.stock = stock;
        binsCreated = binsCreated + 1;
    }
    public boolean take(int amount) {
        if (amount <= 0 || amount > stock) {
            return false;
        }
        stock = stock - amount;
        return true;
    }
    public int getStock() {
        return stock;
    }
    public static int getBinsCreated() {
        return binsCreated;
    }
}
ModificationBin original = new ModificationBin(7);
ModificationBin alias = new ModificationBin(7);
System.out.println("Take 2: " + alias.take(2));
System.out.println("Original: " + original.getStock());
System.out.println("Alias: " + alias.getStock());
System.out.println("Bins: " + ModificationBin.getBinsCreated());
```

Expected output:

```text
Take 2: true
Original: 7
Alias: 5
Bins: 2
```

Common error: Assuming the name alias guarantees a shared object after its initializer changes. Creating a second object without counting its constructor. Expecting a call on one independent bin to change both stocks. Replaying the counted fixture without restarting the kernel first.

**Additional test: `Separate bins, then take 2 through alias and take 1 through original`.** The two independent stocks finish at 6 and 5; calls do not increment the shared construction count.

```java
class ModificationBin {
    private int stock;
    private static int binsCreated = 0;
    public ModificationBin(int stock) {
        this.stock = stock;
        binsCreated = binsCreated + 1;
    }
    public boolean take(int amount) {
        if (amount <= 0 || amount > stock) {
            return false;
        }
        stock = stock - amount;
        return true;
    }
    public int getStock() {
        return stock;
    }
    public static int getBinsCreated() {
        return binsCreated;
    }
}
ModificationBin original = new ModificationBin(7);
ModificationBin alias = new ModificationBin(7);
System.out.println("Take 2: " + alias.take(2));
System.out.println("Take 1 from original: " + original.take(1));
System.out.println("Original: " + original.getStock());
System.out.println("Alias: " + alias.getStock());
System.out.println("Bins: " + ModificationBin.getBinsCreated());
```

Expected output:

```text
Take 2: true
Take 1 from original: true
Original: 6
Alias: 5
Bins: 2
```

</details>


### Reject an invalid request before changing stock

The displayed draft is intentionally incorrect. Do not run the displayed draft. It subtracts before deciding whether a request is valid. Trace both calls and predict each returned result and stock report. Identify the first statement that breaks the nonnegative-stock rule. Write the complete repaired program in the empty work cell. Check `amount <= 0 || amount > stock` before any subtraction and return false immediately for that invalid case. Only then subtract and return true. Keep the class, constructor, reader, inputs and print statements unchanged. Predict and run the repair. Then change only the two request amounts and their labels from 6 and 4 to 0 and -1; both requests must be rejected while stock stays 4. Run the complete repaired cell for that test. Explain why returning false cannot undo an earlier assignment. Restore the original requests and labels when finished.

This sample is for repair:

```java
class RequestBin {
    private int stock;
    public RequestBin(int stock) {
        this.stock = stock;
    }
    public boolean take(int amount) {
        stock = stock - amount;
        if (amount <= 0 || stock < 0) {
            return false;
        }
        return true;
    }
    public int getStock() {
        return stock;
    }
}
RequestBin bin = new RequestBin(4);
System.out.println("Take 6: " + bin.take(6));
System.out.println("Stock: " + bin.getStock());
System.out.println("Take 4: " + bin.take(4));
System.out.println("Stock: " + bin.getStock());
```


| Faulty call | Stock before | Stock after premature subtraction | Returned boolean | Invariant still true? |
|---|---|---|---|---|
| bin.take(6) | | | | |
| bin.take(4) | | | | |


My predicted faulty output:

The first statement that breaks the invariant:

My repaired guard and statement order:

My predicted repaired output:

My actual repaired output:

My predicted and actual zero/negative-request output:

Why returning false does not undo an earlier change:

Why private access does not repair a faulty class method:

<details>
<summary>Show answer</summary>

In the faulty program, take changes stock before deciding whether to reject the request. Taking 6 from 4 stores -2 and then returns false. A later request for 4 subtracts again, stores -6 and returns false again. Returning false does not undo those assignments, and private access alone cannot prevent a badly written method from violating its own rule. The repair first checks amount <= 0 || amount > stock against the unchanged current stock. It returns false immediately for an invalid request. Only an accepted request reaches subtraction and returns true. The request for 6 now leaves stock 4; the exact request for 4 succeeds and leaves zero. Repaired zero and negative requests also return false before any subtraction, preserving stock 4.

```java
class RequestBin {
    private int stock;
    public RequestBin(int stock) {
        this.stock = stock;
    }
    public boolean take(int amount) {
        if (amount <= 0 || amount > stock) {
            return false;
        }
        stock = stock - amount;
        return true;
    }
    public int getStock() {
        return stock;
    }
}
RequestBin bin = new RequestBin(4);
System.out.println("Take 6: " + bin.take(6));
System.out.println("Stock: " + bin.getStock());
System.out.println("Take 4: " + bin.take(4));
System.out.println("Stock: " + bin.getStock());
```

Expected output:

```text
Take 6: false
Stock: 4
Take 4: true
Stock: 0
```

Common error: Leaving subtraction before the rejection guard. Returning false after mutating the object and assuming the change is undone. Rejecting an exact request with amount >= stock. Allowing negative requests to add stock or zero requests to count as successful. Making stock public instead of fixing the operation.

**Additional test: `Zero and negative requests from an initial stock of 4`.** Both nonpositive requests return false before subtraction, so both stock reports stay at 4.

```java
class RequestBin {
    private int stock;
    public RequestBin(int stock) {
        this.stock = stock;
    }
    public boolean take(int amount) {
        if (amount <= 0 || amount > stock) {
            return false;
        }
        stock = stock - amount;
        return true;
    }
    public int getStock() {
        return stock;
    }
}
RequestBin bin = new RequestBin(4);
System.out.println("Take 0: " + bin.take(0));
System.out.println("Stock: " + bin.getStock());
System.out.println("Take -1: " + bin.take(-1));
System.out.println("Stock: " + bin.getStock());
```

Expected output:

```text
Take 0: false
Stock: 4
Take -1: false
Stock: 4
```

</details>


## Independent Practice

### Build two protected seat pools

Define SeatPool with a private int available field and a private static int poolsCreated field initialized to zero. Its public constructor accepts an initially nonnegative available count, stores it in the object and increments the class construction count. Write public boolean reserve(int seats): reject nonpositive requests or requests larger than available by returning false before changing state; otherwise subtract and return true. Add public int getAvailable() and public static int getPoolsCreated() readers. Create morning with 4 seats and evening with 2. Print the result of morning.reserve(3) labeled Reserve 3, then morning.reserve(2) labeled Reserve 2. Print the remaining Morning and Evening counts, then Pools from the static reader. The required lines are `Reserve 3: true`, `Reserve 2: false`, `Morning: 1`, `Evening: 2`, and `Pools: 2`. Plan the state rule and separate field roles, write your own complete program, predict its output, restart the Java kernel, and run the complete work cell once. Restart before each further attempt at this counted fixture. Explain what belongs to each pool and what is shared. Keep constructor input counts nonnegative; this constructor assumes that rule and does not validate negative starting counts.

My instance field and its invariant:

My constructor precondition and initialization:

My shared static field and when it changes:

My rejection guard and update order:

My public readers and their roles:

My predicted complete output:

My actual complete output:

Why the rejected reservation leaves Morning unchanged:

Why Evening stays separate:

Why reservation calls do not change Pools:

I restarted the Java kernel before this counted run:


### Test the boundary and unchanged state after rejection

Test every scenario in the table using your complete SeatPool program. Keep the class unchanged. Change only the constructor count or the two reservation amounts and their matching output labels as stated. Before every scenario and every retry, restart the Java kernel. Then run only your complete class and caller cell once. This initializes the static count as well as creating fresh morning and evening objects; repeating an identical class definition without a restart can preserve the earlier counter. Before each run, predict all five output lines; afterward record the actual results and remaining seat counts. Check the exact remaining seat, zero requests, excessive requests, negative requests and an initially empty pool. For rejected requests, verify both false and unchanged available seats. Explain why Evening and the number of constructed pools have their results in each scenario. Repair any mismatch and repeat all cases. Restore morning 4, evening 2, requests 3 then 2, and the exact original output at the end. Do not test a negative constructor count or claim that this constructor rejects one.

| Initial Morning / Evening | Requests to Morning, in order | Predicted five output lines | Actual five output lines | Match or repair |
|---|---|---|---|---|
| 4 / 2 | 3 then 2 | | | |
| 4 / 2 | 3 then 1 | | | |
| 4 / 2 | 0 then 5 | | | |
| 4 / 2 | -1 then 5 | | | |
| 0 / 2 | 3 then 2 | | | |


My correction and the cases I repeated:

Why the exact remaining request succeeds:

How zero and negative requests differ from allowed requests:

Evidence that an excessive request leaves state unchanged:

Why an initially empty pool is valid:

Why Evening remains a separate instance:

Why the construction count is not the number of reservation calls:

Why I restart the kernel before running the complete class and caller for each test:

My actual output after restoring the contracted program:


<details>
<summary>Show answer</summary>

SeatPool keeps each object’s available field private and stores one shared poolsCreated field in the class. Each constructor initializes one nonnegative seat count and increments that shared count. The public reserve method rejects nonpositive or excessive requests before changing available. Morning starts at 4, accepts 3 and becomes 1; the later request for 2 is rejected without changing that 1. Evening is a separately created object and stays at 2. Two constructors ran, so the static reader reports 2 pools. Encapsulation gives callers controlled operations; the explicit guard preserves the available >= 0 state invariant. Constructor inputs are a caller precondition, not a negative-input validation implemented by this constructor. After reserving 3, an exact request for the last seat succeeds and leaves zero. A zero request and an excessive request both fail without changing the initial 4 seats. A negative request must also fail without adding seats. A pool explicitly created with zero seats is valid; every positive request from it fails and leaves zero. The separate evening pool stays at 2 in all supplied cases, and the construction count remains 2 because reservations do not construct objects. Restart the Java kernel before each case, then run the full class and caller once. Identical IJava class redefinitions can preserve static fields; replaying that definition is not a reliable counter reset. Each hidden program is therefore checked in its own fresh Java kernel.

```java
class SeatPool {
    private int available;
    private static int poolsCreated = 0;
    public SeatPool(int available) {
        this.available = available;
        poolsCreated = poolsCreated + 1;
    }
    public boolean reserve(int seats) {
        if (seats <= 0 || seats > available) {
            return false;
        }
        available = available - seats;
        return true;
    }
    public int getAvailable() {
        return available;
    }
    public static int getPoolsCreated() {
        return poolsCreated;
    }
}
SeatPool morning = new SeatPool(4);
SeatPool evening = new SeatPool(2);
System.out.println("Reserve 3: " + morning.reserve(3));
System.out.println("Reserve 2: " + morning.reserve(2));
System.out.println("Morning: " + morning.getAvailable());
System.out.println("Evening: " + evening.getAvailable());
System.out.println("Pools: " + SeatPool.getPoolsCreated());
```

Expected output:

```text
Reserve 3: true
Reserve 2: false
Morning: 1
Evening: 2
Pools: 2
```

Common error: Using static for available and accidentally sharing all seat counts. Using a separate instance construction counter for each object. Changing available before returning false. Exposing a public field instead of the requested public operations. Counting reservation calls or reference variables as new pools. Claiming the constructor rejects negative values when it does not.

**Additional test: Morning 4/evening 2, reserve 3 then the exact remaining 1.** Both requests succeed; using the final seat is valid and leaves zero without changing Evening or the construction count.

```java
class SeatPool {
    private int available;
    private static int poolsCreated = 0;
    public SeatPool(int available) {
        this.available = available;
        poolsCreated = poolsCreated + 1;
    }
    public boolean reserve(int seats) {
        if (seats <= 0 || seats > available) {
            return false;
        }
        available = available - seats;
        return true;
    }
    public int getAvailable() {
        return available;
    }
    public static int getPoolsCreated() {
        return poolsCreated;
    }
}
SeatPool morning = new SeatPool(4);
SeatPool evening = new SeatPool(2);
System.out.println("Reserve 3: " + morning.reserve(3));
System.out.println("Reserve 1: " + morning.reserve(1));
System.out.println("Morning: " + morning.getAvailable());
System.out.println("Evening: " + evening.getAvailable());
System.out.println("Pools: " + SeatPool.getPoolsCreated());
```

Expected output:

```text
Reserve 3: true
Reserve 1: true
Morning: 0
Evening: 2
Pools: 2
```

**Additional test: Morning 4/evening 2, reserve 0 then excessive 5.** Neither request is accepted. Both checks leave the original four morning seats unchanged.

```java
class SeatPool {
    private int available;
    private static int poolsCreated = 0;
    public SeatPool(int available) {
        this.available = available;
        poolsCreated = poolsCreated + 1;
    }
    public boolean reserve(int seats) {
        if (seats <= 0 || seats > available) {
            return false;
        }
        available = available - seats;
        return true;
    }
    public int getAvailable() {
        return available;
    }
    public static int getPoolsCreated() {
        return poolsCreated;
    }
}
SeatPool morning = new SeatPool(4);
SeatPool evening = new SeatPool(2);
System.out.println("Reserve 0: " + morning.reserve(0));
System.out.println("Reserve 5: " + morning.reserve(5));
System.out.println("Morning: " + morning.getAvailable());
System.out.println("Evening: " + evening.getAvailable());
System.out.println("Pools: " + SeatPool.getPoolsCreated());
```

Expected output:

```text
Reserve 0: false
Reserve 5: false
Morning: 4
Evening: 2
Pools: 2
```

**Additional test: Morning 4/evening 2, reserve -1 then excessive 5.** A negative request is rejected without adding a seat, so the later request for 5 is still excessive and Morning stays at 4.

```java
class SeatPool {
    private int available;
    private static int poolsCreated = 0;
    public SeatPool(int available) {
        this.available = available;
        poolsCreated = poolsCreated + 1;
    }
    public boolean reserve(int seats) {
        if (seats <= 0 || seats > available) {
            return false;
        }
        available = available - seats;
        return true;
    }
    public int getAvailable() {
        return available;
    }
    public static int getPoolsCreated() {
        return poolsCreated;
    }
}
SeatPool morning = new SeatPool(4);
SeatPool evening = new SeatPool(2);
System.out.println("Reserve -1: " + morning.reserve(-1));
System.out.println("Reserve 5: " + morning.reserve(5));
System.out.println("Morning: " + morning.getAvailable());
System.out.println("Evening: " + evening.getAvailable());
System.out.println("Pools: " + SeatPool.getPoolsCreated());
```

Expected output:

```text
Reserve -1: false
Reserve 5: false
Morning: 4
Evening: 2
Pools: 2
```

**Additional test: Initially empty Morning 0/evening 2, reserve 3 then 2.** Zero is an allowed initial count. Both positive requests fail; Morning stays at 0, Evening at 2, and both constructions still count.

```java
class SeatPool {
    private int available;
    private static int poolsCreated = 0;
    public SeatPool(int available) {
        this.available = available;
        poolsCreated = poolsCreated + 1;
    }
    public boolean reserve(int seats) {
        if (seats <= 0 || seats > available) {
            return false;
        }
        available = available - seats;
        return true;
    }
    public int getAvailable() {
        return available;
    }
    public static int getPoolsCreated() {
        return poolsCreated;
    }
}
SeatPool morning = new SeatPool(0);
SeatPool evening = new SeatPool(2);
System.out.println("Reserve 3: " + morning.reserve(3));
System.out.println("Reserve 2: " + morning.reserve(2));
System.out.println("Morning: " + morning.getAvailable());
System.out.println("Evening: " + evening.getAvailable());
System.out.println("Pools: " + SeatPool.getPoolsCreated());
```

Expected output:

```text
Reserve 3: false
Reserve 2: false
Morning: 0
Evening: 2
Pools: 2
```

</details>


## Summary

Encapsulation exposes intended operations while keeping state behind access controls. An invariant states the rule those operations must preserve. A copied reference can reach the same object, so mutation through an alias or parameter is visible to other references. Reassigning a local parameter does not replace the caller’s variable. Static fields belong to the class; instance fields belong separately to its objects.

Close the answers and trace an accepted request, a rejected request, a shared-object change and a local reference reassignment.


## Reflection

Describe a rule for a real object in your field, such as remaining seats or available equipment. Propose one operation and explain its result for an accepted input, a rejected input, and an exact boundary. Identify which values should belong to each object.

**My design and explanation:**

Next, you will use private array storage and controlled operations to build a collection whose logical size can grow.


## Supplemental Reading

- [More on Java classes](https://dev.java/learn/classes-objects/more-on-classes/) covers access control and class versus instance members.
- [Calling Java methods and constructors](https://dev.java/learn/classes-objects/calling-methods-constructors/) explains why both primitive and reference argument values are copied.
